<table style="width: 100%; margin-left: auto; margin-right: auto; background-color: white;">
<tbody>
<tr>

<td style="text-align: center; width: 12%;"><img src=img/brasil-brasao.png alt="" /></td>

<td style="width: 68%;">
<h2 style="text-align: center; margin-top: 1%;">Rep&uacute;blica Federativa do Brasil</h2>
<h2 style="text-align: center; margin-top: 1%;">Minist&eacute;rio da Educa&ccedil;&atilde;o</h2>
<h2 style="text-align: center; margin-top: 1%;">Universidade Federal do Amazonas</h2>
<h2 style="text-align: center; margin-top: 1%;">Instituto de Computa&ccedil;&atilde;o</h2>
</td>

<td style="width: 12%;"><img src="img/UFAM-logo.png" alt="" /></td>

</tr>
</tbody>
</table>

---

# Ciência de Dados

## Trabalho Prático I: Exploração de Dados, v 1.0

_Professor Dr. Marco Cristo_


**Alunos:** `Nome Sobrenome, Nome Sobrenome e Nome Sobrenome` <br>
**E-mail:** `nome.sobrenome@icomp.ufam.edu.br, nome.sobrenome@icomp.ufam.edu.br e nome.sobrenome@icomp.ufam.edu.br`


#### Sobre os dados

Este trabalho usa os microdados da Amostra do Censo Demográfico 2010 (IBGE), com foco no tema **Trabalho e Rendimento**. Cada equipe (este trabalho é para trios) trabalha sobre a população do seu **município** (especificado para a equipe) e a compara com o restante do país. Consulte o notebook `Dicionário de dados.ipynb` para a descrição de cada variável.

**Atenção**: por motivos didáticos, a base de dados contém alguns nulos e inconsistências propositalmente introduzidas (além dos nulos que já ocorrem naturalmente por questões de desenho da pesquisa do IBGE) — parte do seu trabalho na seção de Qualidade de Dados é justamente encontrá-los e tratá-los.


#### Preliminares


In [1]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

import sklearn
import sys
import os

In [2]:
print("Python", sys.version)
print("--------")
print("Pandas:", pd.__version__)
print("Numpy:", np.__version__)
print("SKLearn:", sklearn.__version__)

Python 3.13.5 (main, Aug 10 2026, 12:06:59) [GCC 14.2.0]
--------
Pandas: 3.0.3
Numpy: 2.2.6
SKLearn: 1.9.0


Escolha o arquivo de dados com o seu nome no diretório `data`.


In [3]:
DATA_PQT = 'data/censo2010_pessoas.parquet'
MUNICIPIO = 'São Bernardo do Campo' # Use o que foi especificado para sua equipe
SIGLA_UF = 'SP'      # Sigla da UF do município acima 

In [8]:
# Caminho do arquivo
df_censo = pd.read_parquet(DATA_PQT)
df_censo = df_censo[df_censo['SIGLA_UF']== 'SP']
df_censo[df_censo['NOME_MUNICIPIO']== 'São Bernardo do Campo']



        COD_UF COD_MUNICIPIO_UF AREA_PONDERACAO    V0300  PESO_AMOSTRAL SEXO  \
3168586     35            48708   3548708005015   832862      11.368428    2   
3168587     35            48708   3548708005018   517014      18.232156    2   
3168588     35            48708   3548708005002  1126308      13.948775    2   
3168589     35            48708   3548708005007  3656115      11.438531    1   
3168590     35            48708   3548708005003  2226132      19.081957    1   
...        ...              ...             ...      ...            ...  ...   
3206174     35            48708   3548708005006  2010843      17.364349    2   
3206175     35            48708   3548708005001  2585687      20.127779    1   
3206176     35            48708   3548708005013  4382263      25.840239    2   
3206177     35            48708   3548708005013  4043152      20.820199    1   
3206178     35            48708   3548708005004  5560576      10.397420    1   

         IDADE COR_RACA NIVEL_INSTRUCAO

---

#### Qualidade de dados


<div class="alert alert-info">
    <b>(0,2) 1</b> - Quais colunas possuem campos nulos em <code>df_censo</code>? Ordene-as da que tem mais nulos para a que tem menos.
</div>


<div class="alert alert-primary">
<b>(0,1) 1.1</b> - Várias dessas colunas têm nulos pelo mesmo motivo: o questionário do Censo pula perguntas que não se aplicam a quem responde (ex.: perguntas sobre o trabalho não fazem sentido para quem não trabalha). Escolha três colunas com muitos nulos e explique, olhando o dicionário de dados, por que elas têm nulos "estruturais" (que não são erro, mas sim ausência de aplicabilidade da pergunta).
</div>


<div class="alert alert-info">
<b>(0,25) 2</b> - Investigue a relação entre <code>IDADE</code> e os nulos em <code>RENDA_TOTAL</code>. Existe um padrão claro? Baseado nisso, decida um critério de idade mínima para manter no conjunto de dados e remova as instâncias fora desse critério.
</div>

<i>Dica</i>: compare a distribuição de <code>IDADE</code> entre quem tem <code>RENDA_TOTAL</code> nulo e quem não tem.


<div class="alert alert-primary">
<b>(0,15) 2.1</b> - Depois de aplicar o filtro acima, ainda restam nulos em <code>RENDA_TOTAL</code>? Se sim, quantos, e o que isso sugere?
</div>


<div class="alert alert-info">
<b>(0,3) 3</b> - Verifique se há valores de <code>IDADE</code> ou <code>SEXO</code> fora do domínio esperado pelo dicionário de dados (idades implausíveis, códigos de sexo diferentes de 1/2). Trate essas inconsistências.
</div>


<div class="alert alert-info">
<b>(0,2) 4</b> - Verifique se há linhas completamente duplicadas em <code>df_censo</code>. Remova-as.
</div>


<div class="alert alert-info">
<b>(0,3) 5</b> - Compare <code>RENDA_TOTAL</code> com <code>RENDA_TODOS_TRABALHOS</code> (que deveria ser sempre menor ou igual, já que a primeira soma a segunda com outras rendas). Identifique casos em que <code>RENDA_TOTAL</code> é muito maior que <code>RENDA_TODOS_TRABALHOS</code> (ex.: mais de 10x mais alta) -- um padrão consistente com erro de digitação (ex.: vírgula decimal digitada no lugar errado). Corrija esses casos.
</div>


<div class="alert alert-info">
<b>(0,5) 6</b> - Ainda restam nulos em <code>RENDA_TOTAL</code> (o resíduo identificado na questão 2.1) e em <code>COR_RACA</code>, <code>NIVEL_INSTRUCAO</code> e <code>RENDA_DOMICILIAR_PER_CAPITA</code>. Decida e aplique uma estratégia de tratamento para cada uma, justificando sua escolha.
</div>

<i>Dica</i>: nem toda coluna precisa do mesmo tratamento -- pense se faz mais sentido remover a linha, imputar um valor, ou criar uma categoria "não informado".


---

#### Análise e visualização de dados


Para as próximas questões, criamos uma coluna auxiliar `GRUPO` que marca se a pessoa está no seu município de interesse ou no "resto do país".


In [5]:
df_censo['GRUPO'] = np.where(df_censo.NOME_MUNICIPIO==MUNICIPIO, MUNICIPIO, 'Resto do país')

<div class="alert alert-info">
<b>(0,35) 7</b> - Compare a distribuição de <code>RENDA_TOTAL</code> entre homens e mulheres, tanto no seu município quanto no resto do país. Homens ganham significativamente mais que mulheres (confiança de 95%)? Use boxplots e um teste não paramétrico unicaudal apropriado.
</div>


<div class="alert alert-primary">
<b>(0,2) 7.1</b> - Explique os resultados obtidos.
</div>


<div class="alert alert-info">
<b>(0,4) 8</b> - Compare a distribuição de <code>RENDA_TOTAL</code> por cor/raça (<code>COR_RACA</code>), tanto no seu município quanto no resto do país. Use um gráfico de violino e um gráfico de barras com intervalo de confiança. Qual a melhor visualização e porque?
</div>


<div class="alert alert-primary">
<b>(0,25) 8.1</b> - Explique os resultados obtidos.
</div>


<div class="alert alert-info">
<b>(0,35) 9</b> - Trace a renda média por faixa etária (crie faixas de 10 em 10 anos, aproximadamente) tanto para o seu município quanto para o resto do país, usando um gráfico de linhas com intervalo de confiança (95%). Isso corresponde ao "ciclo de vida" da renda de uma pessoa. As diferenças de renda média entre faixas etárias vizinhas são estatisticamente significativas em toda a curva ou o tamanho da amostra é pequeno demais para sustentar essa leitura em algum trecho?
</div>


<div class="alert alert-primary">
<b>(0,3) 9.1</b> - Explique os resultados obtidos.
</div>


<div class="alert alert-info">
<b>(0,35) 10</b> - Qual a relação entre o nível de instrução (<code>NIVEL_INSTRUCAO</code>) e a renda, tanto no seu município quanto no resto do país? Mostre uma tabela dinâmica (pivot table) com a renda média por nível de instrução e grupo, e calcule a correlação de Spearman entre nível de instrução (tratado como ordinal) e renda em cada grupo.
</div>


<div class="alert alert-primary">
<b>(0,2) 10.1</b> - Explique os resultados obtidos.
</div>


<div class="alert alert-info">
<b>(0,3) 11</b> - Qual a renda média de acordo com a posição na ocupação (<code>POSICAO_OCUPACAO</code>), tanto no seu município quanto no resto do país?
</div>


<div class="alert alert-primary">
<b>(0,2) 11.1</b> - Explique os resultados obtidos.
</div>


<div class="alert alert-info">
<b>(0,35) 12</b> - Combine cor/raça e sexo em uma única tabela dinâmica de renda média, considerando apenas o seu município. O que se observa sobre desigualdade "interseccional" (por raça e por gênero simultaneamente)?
</div>


<div class="alert alert-primary">
<b>(0,25) 12.1</b> - Explique os resultados obtidos.
</div>


---

#### Transformação e engenharia de atributos


<div class="alert alert-info">
<b>(0,25) 13</b> - Examine a cardinalidade (número de valores distintos) de <code>COD_OCUPACAO</code> e <code>COD_ATIVIDADE_ECON</code>, comparando com a de outras variáveis categóricas da base (ex.: <code>POSICAO_OCUPACAO</code>, <code>SUBGRUPO_OCUPACAO</code>, <code>NIVEL_INSTRUCAO</code>, <code>COR_RACA</code>).
</div>


<div class="alert alert-primary">
<b>(0,25) 13.1</b> - O que essa cardinalidade implica para o uso de <code>COD_OCUPACAO</code> e <code>COD_ATIVIDADE_ECON</code> como atributos preditivos (ex.: ao aplicar one-hot encoding)? Cite ao menos uma estratégia de codificação alternativa que poderia mitigar o problema, caso decidam usar essas variáveis no TP2.
</div>


<div class="alert alert-info">
<b>(0,3) 14</b> - A partir de <code>IDADE</code>, crie a coluna <code>FAIXA_ETARIA</code> com os seguintes intervalos (mantenha esse nome de coluna e esses intervalos -- eles serão reaproveitados no TP2):
<ul>
<li>[10..15)</li>
<li>[15..18)</li>
<li>[18..25)</li>
<li>[25..35)</li>
<li>[35..45)</li>
<li>[45..55)</li>
<li>[55..65)</li>
<li>[65..120]</li>
</ul>
</div>


<div class="alert alert-info">
<b>(0,25) 15</b> - A partir de <code>RENDA_TOTAL_SM</code> (renda em número de salários mínimos, já calculada pelo IBGE), crie a coluna <code>FAIXA_RENDA</code> usando os cortes oficiais do IBGE (mantenha esse nome de coluna e esses cortes -- essa será a variável-alvo da tarefa de <b>classificação</b> no TP2):
<ul>
<li>Sem rendimento</li>
<li>Até 1/4 SM</li>
<li>Mais de 1/4 até 1/2 SM</li>
<li>Mais de 1/2 até 1 SM</li>
<li>Mais de 1 até 2 SM</li>
<li>Mais de 2 até 3 SM</li>
<li>Mais de 3 até 5 SM</li>
<li>Mais de 5 até 10 SM</li>
<li>Mais de 10 até 15 SM</li>
<li>Mais de 15 até 20 SM</li>
<li>Mais de 20 SM</li>
</ul>
</div>


<div class="alert alert-primary">
<b>(0,25) 15.1</b> - Essa variável está balanceada entre as classes? Isso é um problema para a tarefa de classificação que virá no TP2? O que poderia ser feito a respeito?
</div>


<div class="alert alert-info">
<b>(0,7) 16</b> - Que atributos são os mais importantes para prever <code>FAIXA_RENDA</code>, considerando correlação absoluta, ANOVA, qui-quadrado, informação mútua e a importância de variáveis segundo uma floresta aleatória? Considere apenas os candidatos da sua cidade de interesse (isso garante que os resultados variem entre equipes).
<br><i>Dica</i>: Para facilitar a visualização, coloque todas as métricas em um único dataframe. Depois normalize-as para que todas fiquem entre 0 e 1. Obtenha então uma média agregada das métricas normalizadas e use essa média para ordenar as variáveis da mais à menos importante.
</div>


<div class="alert alert-primary">
<b>(0,5) 16.1</b> - Explique os resultados obtidos. As cinco métricas concordam entre si? Alguma posição no ranking te surpreendeu, para cima ou para baixo? Isso muda sua decisão sobre quais atributos usar no TP2?
</div>


---

#### Comunicação e storytelling


<div class="alert alert-info">
<b>(1,5) 17</b> - Produza um vídeo de até 3 minutos, com narração (voiceover), contando uma única história a partir dos achados do TP1, para uma audiência do seu município de interesse (ex.: um gestor público local, um jornalista da cidade). Escolha 2 achados das questões 7 a 12 (que comparam seu município com o resto do país) que juntos sustentem essa mensagem central (a "big idea"); não tente encaixar todos os achados do trabalho, isso é ruído, não sinal. Como a audiência e os dados são específicos da sua cidade, a história precisa fazer sentido para o seu recorte. Aplique os princípios de storytelling with data vistos no curso: contexto da audiência, uma mensagem central clara, remoção de excesso visual (declutter), uso de foco/destaque (cor, texto, posição) para guiar o olhar e um arco narrativo (contexto -> tensão/descoberta -> conclusão). <b>Vídeos com mais de 3 minutos não serão avaliados</b>. A equipe é responsável por conferir a duração antes de entregar.

Poste o vídeo no YouTube (não listado) ou Google Drive (com compartilhamento "qualquer pessoa com o link pode assistir") e inclua o link nesta célula. Não envie o arquivo de vídeo diretamente pelo Classroom. Exporte em resolução moderada (720p já é suficiente) para manter o arquivo pequeno.
<br><i>Dica</i>: o vídeo costuma ser a parte com a curva de aprendizado mais alta pra quem nunca gravou/editou narração antes. Não deixe pra fazer na última semana. Uma boa prática é, na metade do prazo, já ter escolhido os 2 achados e um rascunho do roteiro, mesmo que o resto do TP1 ainda não esteja pronto.

</div>


<div class="alert alert-primary">
<b>(0,5) 17.1</b> - Em 3-4 frases, justifique as escolhas de design que você fez: por que esse recorte de achados, qual é a "big idea", e qual elemento de storytelling with data você usou mais deliberadamente?
</div>
